# Check DEM structure for a cultivation circuit

Build a d1=3, d2=7 end-to-end cultivation circuit with unitary injection
and uniform depolarizing noise at p=0.001, then inspect the DEM.

In [1]:
import sys
sys.path.insert(0, '../src')

import stim
import numpy as np
import cultiv
import gen

## 1. Build the ideal circuit, then add noise

In [2]:
D1           = 3          # color code distance (injection + cultivation)
D2           = 7          # surface code distance (escape target)
BASIS        = 'Y'        # magic state basis
INJECT_STYLE = 'unitary'  # injection protocol
R_GROWING    = D1         # rounds during the growing/escape stage (convention: = D1)
R_END        = 0          # rounds of idle matchable-code after escape
P            = 1e-3       # physical error rate

print(f'd1={D1}, d2={D2}, basis={BASIS}, inject={INJECT_STYLE}, r_growing={R_GROWING}, p={P}')

d1=3, d2=7, basis=Y, inject=unitary, r_growing=3, p=0.001


In [3]:
# Build ideal (noiseless) circuit
ideal_circuit = cultiv.make_end2end_cultivation_circuit(
    dcolor=D1,
    dsurface=D2,
    basis=BASIS,
    r_growing=R_GROWING,
    r_end=R_END,
    inject_style=INJECT_STYLE,
)

# Apply uniform depolarizing noise (skipping MPP boundary instructions)
noise_model  = gen.NoiseModel.uniform_depolarizing(P)
noisy_circuit = noise_model.noisy_circuit_skipping_mpp_boundaries(ideal_circuit)

print('Ideal circuit qubits :', ideal_circuit.num_qubits)
print('Ideal circuit ticks  :', ideal_circuit.num_ticks)
print('Ideal circuit measures:', ideal_circuit.num_measurements)

Ideal circuit qubits : 102
Ideal circuit ticks  : 79
Ideal circuit measures: 265


## 2. Extract the DEM (undecomposed — true hyperedges kept)

In [4]:
decompose_errors = True
dem = noisy_circuit.detector_error_model(
    decompose_errors=decompose_errors,   # keep true hyperedges (needed for LSD / JIT)
    ignore_decomposition_failures=decompose_errors,  # don't fail if some errors can't be decomposed into detectors
    flatten_loops=True,       # unroll repeat blocks -> concrete detector IDs
)

print(f'Number of detectors  : {dem.num_detectors}')
print(f'Number of observables: {dem.num_observables}')

# Count instruction types
from collections import Counter
type_counts = Counter(inst.type for inst in dem)
print(f'Instruction types    : {dict(type_counts)}')

Number of detectors  : 239
Number of observables: 1
Instruction types    : {'error': 4675, 'detector': 239, 'shift_detectors': 6}


In [33]:
print(dem)

error(0.002993251351171128359) D0
error(0.004251500183731311486) D0 D17 L0
error(0.0001333866998761607556) D0 D17 L0 ^ D1 ^ D2 D18
error(6.669779853440971351e-05) D0 D17 L0 ^ D1 ^ D2 D18 ^ D4 D16 L0 ^ D7 D22
error(6.669779853440971351e-05) D0 D17 L0 ^ D1 ^ D2 D18 ^ D4 ^ D5 D7 ^ D6 D16 L0
error(0.004449711097248765755) D0 D17 L0 ^ D1 ^ D2 D18 ^ D4 ^ D5 D22 ^ D6 D16 L0
error(6.669779853440971351e-05) D0 D17 L0 ^ D1 ^ D2 D18 ^ D4 ^ D5 D22 ^ D15 L0 ^ D16
error(6.669779853440971351e-05) D0 D17 L0 ^ D1 ^ D2 D18 ^ D4 ^ D6 D7 ^ D16 ^ D20 L0
error(0.0001333866998761607556) D0 D17 L0 ^ D1 ^ D2 D18 ^ D4 ^ D6 D16 L0 ^ D20 D22
error(6.669779853440971351e-05) D0 D17 L0 ^ D1 ^ D2 D18 ^ D5 D7 ^ D20 D22
error(0.0001333866998761607556) D0 D17 L0 ^ D1 ^ D2 D18 ^ D5 D20
error(0.0001333866998761607556) D0 D17 L0 ^ D1 ^ D2 D18 ^ D6 D7 ^ D15 D16 ^ D22 L0
error(6.669779853440971351e-05) D0 D17 L0 ^ D1 ^ D2 D18 ^ D6 D7 ^ D15 L0 ^ D22 L0
error(0.004647997324384119458) D0 D17 L0 ^ D1 ^ D2 D18 ^ D7 D22
error(6.66

In [34]:
# Show first 20 error instructions
print('=== First 20 error instructions ===')
shown = 0
for inst in dem:
    if inst.type == 'error':
        tgts = inst.targets_copy()
        det_ids = [t.val for t in tgts if t.is_relative_detector_id()]
        obs_ids = [t.val for t in tgts if t.is_logical_observable_id()]
        print(f'  p={inst.args_copy()[0]:.4e}  detectors={det_ids}  observables={obs_ids}')
        shown += 1
        if shown >= 20:
            break

=== First 20 error instructions ===
  p=2.9933e-03  detectors=[0]  observables=[]
  p=4.2515e-03  detectors=[0, 17]  observables=[0]
  p=1.3339e-04  detectors=[0, 17, 1, 2, 18]  observables=[0]
  p=6.6698e-05  detectors=[0, 17, 1, 2, 18, 4, 16, 7, 22]  observables=[0, 0]
  p=6.6698e-05  detectors=[0, 17, 1, 2, 18, 4, 5, 7, 6, 16]  observables=[0, 0]
  p=4.4497e-03  detectors=[0, 17, 1, 2, 18, 4, 5, 22, 6, 16]  observables=[0, 0]
  p=6.6698e-05  detectors=[0, 17, 1, 2, 18, 4, 5, 22, 15, 16]  observables=[0, 0]
  p=6.6698e-05  detectors=[0, 17, 1, 2, 18, 4, 6, 7, 16, 20]  observables=[0, 0]
  p=1.3339e-04  detectors=[0, 17, 1, 2, 18, 4, 6, 16, 20, 22]  observables=[0, 0]
  p=6.6698e-05  detectors=[0, 17, 1, 2, 18, 5, 7, 20, 22]  observables=[0]
  p=1.3339e-04  detectors=[0, 17, 1, 2, 18, 5, 20]  observables=[0]
  p=1.3339e-04  detectors=[0, 17, 1, 2, 18, 6, 7, 15, 16, 22]  observables=[0, 0]
  p=6.6698e-05  detectors=[0, 17, 1, 2, 18, 6, 7, 15, 22]  observables=[0, 0, 0]
  p=4.6480e-03  

## 3. Detector coordinates

Each detector gets spacetime coordinates `[x, y, t, extra1, extra2]`.
The `t` coordinate (index 2) identifies the round — useful for distinguishing
spacelike vs timelike faults.

In [35]:
coords = dem.get_detector_coordinates()
# coords: dict[int, list[float]]  ->  {detector_id: [x, y, t, ...]}
# print(coords)

print(f'Total detectors with coordinates: {len(coords)}')
print()

# Show first 20 detectors
print('=== First 20 detector coordinates ===')
for det_id in sorted(coords)[:]:
    c = coords[det_id]
    print(f'  D{det_id:4d}  coords={c}')

Total detectors with coordinates: 239

=== First 20 detector coordinates ===
  D   0  coords=[2.0, 0.0, 0.0]
  D   1  coords=[4.0, 1.0, 0.0]
  D   2  coords=[2.0, 2.0, 0.0]
  D   3  coords=[1.0, 3.0, 0.0]
  D   4  coords=[1.0, 0.0, 0.0]
  D   5  coords=[3.0, 1.0, 0.0]
  D   6  coords=[1.0, 2.0, 0.0]
  D   7  coords=[2.142857142857143, 1.0, 1.0, -1.0, -9.0]
  D   8  coords=[4.0, 1.0, 2.0]
  D   9  coords=[3.0, 1.0, 2.0]
  D  10  coords=[2.0, 1.0, 2.0]
  D  11  coords=[1.0, 0.0, 2.0]
  D  12  coords=[2.0, 2.0, 2.0]
  D  13  coords=[0.0, 1.0, 2.0]
  D  14  coords=[3.0, 5.0, 4.0, 0.0, 6.0]
  D  15  coords=[1.125, 0.125, 4.0, -1.0, -9.0]
  D  16  coords=[1.25, 1.9375, 4.0, -1.0, -9.0]
  D  17  coords=[1.875, 0.125, 4.0, -1.0, -9.0]
  D  18  coords=[2.0, 1.9375, 4.0, -1.0, -9.0]
  D  19  coords=[3.0, -2.0, 4.0, 3.0, 7.0]
  D  20  coords=[3.0, 0.9375, 4.0, -1.0, -9.0]
  D  21  coords=[3.0, 3.0, 4.0, 0.0, 6.0]
  D  22  coords=[3.75, 0.9375, 4.0, -1.0, -9.0]
  D  23  coords=[5.0, -4.0, 4.0, 3.0

In [36]:
# Summarise the time-coordinate distribution
# (how many detectors fire in each round)
from collections import defaultdict

t_coord_index = 2  # coords[det_id][2] is the time coordinate
round_counts = defaultdict(int)
for det_id, c in coords.items():
    if len(c) > t_coord_index:
        round_counts[c[t_coord_index]] += 1

print('Detectors per time coordinate (round):')
for t in sorted(round_counts):
    print(f'  t={t:.1f}  ->  {round_counts[t]} detectors')

Detectors per time coordinate (round):
  t=0.0  ->  7 detectors
  t=1.0  ->  1 detectors
  t=2.0  ->  6 detectors
  t=4.0  ->  26 detectors
  t=5.0  ->  50 detectors
  t=6.0  ->  50 detectors
  t=7.0  ->  50 detectors
  t=8.0  ->  49 detectors


## 4. Parse DEM into H, O, probs, hyperedges

Preview of what `dem_to_pcm` will do.

In [37]:
n_detectors   = dem.num_detectors
n_observables = dem.num_observables

error_insts = [inst for inst in dem if inst.type == 'error']

# -----------------------------------------------------------------------
# Parse each error instruction into one or more fault columns.
#
# decompose_errors=False: each instruction is one hyperedge, no ^ present.
#   e.g.  error(p) D0 D3 D7 L0
#
# decompose_errors=True: an instruction may contain ^ separator targets
# that split it into decomposed MWPM-compatible pieces.
#   e.g.  error(p) D0 D3 L0 ^ D1 ^ D3 D7
#                  ─────────   ──   ──────
#                  piece 0     p1   piece 2
#
# piece 0 is the "primary" edge (the original fault mechanism).
# pieces 1.. are the additional decomposed edges stim inferred.
# Each piece becomes its own column in H with the same probability p.
#
# When decompose_errors=False there are no ^ so every instruction produces
# exactly one piece (piece 0 = the whole instruction), and behaviour is
# identical to the simple loop.
# -----------------------------------------------------------------------

# First pass: count total columns (one per piece across all instructions)
all_pieces = []   # list of (prob, list[target]) — one entry per column
for inst in error_insts:
    p = inst.args_copy()[0]
    # Split targets at ^ separator targets into pieces
    pieces = []
    current = []
    for t in inst.targets_copy():
        if t.is_separator():
            pieces.append(current)
            current = []
        else:
            current.append(t)
    pieces.append(current)
    for piece in pieces:
        all_pieces.append((p, piece))

n_faults = len(all_pieces)

H          = np.zeros((n_detectors,  n_faults), dtype=np.uint8)
O          = np.zeros((n_observables, n_faults), dtype=np.uint8)
probs      = np.zeros(n_faults, dtype=np.float64)
hyperedges = []   # list[frozenset[int]] — detector IDs per fault column

for j, (p, targets) in enumerate(all_pieces):
    probs[j] = p
    det_set = set()
    for t in targets:
        if t.is_relative_detector_id():
            H[t.val, j] = 1
            det_set.add(t.val)
        elif t.is_logical_observable_id():
            O[t.val, j] = 1
    hyperedges.append(frozenset(det_set))

print(f'decompose_errors     : {decompose_errors}')
print(f'H shape (n_detectors x n_faults) : {H.shape}')
print(f'O shape (n_obs x n_faults)       : {O.shape}')
print(f'probs range                       : [{probs.min():.3e}, {probs.max():.3e}]')
print()

sizes = Counter(len(e) for e in hyperedges)
print('Hyperedge size distribution (detectors per fault column):')
for sz in sorted(sizes):
    print(f'  |edge|={sz}  ->  {sizes[sz]} faults')

decompose_errors     : True
H shape (n_detectors x n_faults) : (239, 9712)
O shape (n_obs x n_faults)       : (1, 9712)
probs range                       : [6.670e-05, 1.408e-02]

Hyperedge size distribution (detectors per fault column):
  |edge|=0  ->  29 faults
  |edge|=1  ->  3351 faults
  |edge|=2  ->  6296 faults
  |edge|=3  ->  4 faults
  |edge|=4  ->  4 faults
  |edge|=5  ->  16 faults
  |edge|=6  ->  6 faults
  |edge|=7  ->  4 faults
  |edge|=8  ->  1 faults
  |edge|=9  ->  1 faults


In [38]:
# For each fault, check if it is timelike:
# a fault is timelike if its detectors span more than one time coordinate.
n_timelike = 0
n_spacelike = 0
n_boundary = 0  # touches 0 or 1 detectors
max_t_diff = 0
t_diff_distribution = Counter()

for j, edge in enumerate(hyperedges):
    if len(edge) <= 1:
        n_boundary += 1
        continue
    t_vals = set()
    for det_id in edge:
        c = coords.get(det_id, [])
        if len(c) > t_coord_index:
            t_vals.add(c[t_coord_index])
            max_t_diff = max(max_t_diff, max(t_vals) - min(t_vals))
            t_diff_distribution[max(t_vals) - min(t_vals)] += 1
    if len(t_vals) > 1:
        n_timelike += 1
    else:
        n_spacelike += 1

print(f'Spacelike faults (same t): {n_spacelike}')
print(f'Timelike faults (diff t) : {n_timelike}')
print(f'Max t difference in any fault: {max_t_diff:.1f} rounds')
print('t difference distribution (for timelike faults):')
for t_diff in sorted(t_diff_distribution):
    print(f'  t_diff={t_diff:.1f}  ->  {t_diff_distribution[t_diff]} faults')
print(f'Boundary faults (<=1 det): {n_boundary}')
print(f'Total faults             : {n_faults}')

Spacelike faults (same t): 2705
Timelike faults (diff t) : 3627
Max t difference in any fault: 4.0 rounds
t difference distribution (for timelike faults):
  t_diff=0.0  ->  9065 faults
  t_diff=1.0  ->  3384 faults
  t_diff=2.0  ->  13 faults
  t_diff=3.0  ->  54 faults
  t_diff=4.0  ->  265 faults
Boundary faults (<=1 det): 3380
Total faults             : 9712


## Check for ParityCheckMatrices class

In [5]:
from parity_matrix_construct import ParityCheckMatrices

In [6]:
# Build ParityCheckMatrices from the already-constructed dem
pcm = ParityCheckMatrices.from_DEM(dem, decompose=decompose_errors)

In [7]:
pcm.print_stats()

decomposed           : True
n_detectors          : 239
n_observables        : 1
n_faults             : 9712
H shape              : (239, 9712)
L shape              : (1, 9712)
prob range           : [6.670e-05, 1.408e-02]

Hyperedge size distribution (detectors per fault column):
  |edge|= 0  ->  29 faults
  |edge|= 1  ->  3351 faults
  |edge|= 2  ->  6296 faults
  |edge|= 3  ->  4 faults
  |edge|= 4  ->  4 faults
  |edge|= 5  ->  16 faults
  |edge|= 6  ->  6 faults
  |edge|= 7  ->  4 faults
  |edge|= 8  ->  1 faults
  |edge|= 9  ->  1 faults

Boundary faults (<=1 detector) : 3380
Spacelike faults (same t)       : 2705
Timelike faults (diff t)        : 3627

Detectors per round (t coordinate):
  t=0.0  ->  7 detectors
  t=1.0  ->  1 detectors
  t=2.0  ->  6 detectors
  t=4.0  ->  26 detectors
  t=5.0  ->  50 detectors
  t=6.0  ->  50 detectors
  t=7.0  ->  50 detectors
  t=8.0  ->  49 detectors


In [10]:
# pcm.print_H()               # first 60 columns
# pcm.print_H(max_cols=20)    # fewer columns
pcm.print_H(max_col_trunc=False)  # all columns (careful — 4222 wide)
pcm.print_L(max_col_trunc=False)  # all columns (careful — 4222 wide)

H  (239 detectors x 9712 faults)
  D   0 | 111001000010000010000010000001000000100000100001000100000100000100010000010000010000100000100000100000100000100001000001000100001000001000010001000010000100001000010001001000010000100010001010001000100010001000010000100100010010001001001001001001000100010001000100001001000010001000100100010001010010001010101010001001000100100100100001000101001000100000100100001001000010000100100010001000100010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000100001000010000100000000000010000000000010000000000000000000000000000000000000000000000000000000000000000000000000000001010000000000000000000000000000000000000000000000000000000000000000000000000000100000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000